### 套用資料驅動字典 + 全量資料的模板剔除／轉載去重

`3-2_dict_from_data.ipynb` 的 `merged_candidates` 分頁已經從五個年份的標註樣本裡萃取出候選樣板片語。這份 notebook 把這份字典套回**全量** snippet 資料（不是只有抽樣的 300 筆），對每個年份做兩件事：

1. **模板剔除**：字典命中就標記 `is_template = True`，不管出現幾次都算，跟頻率無關（只出現一次也剔除）
2. **轉載去重**：沒命中字典、但「不同文章數」達到 `FREQ_THRESHOLD` 門檻的高頻片段，標記 `is_duplicate = True`——這是真實新聞被多家媒體轉載，不能整批刪掉，只保留**最早出現**的那篇文章（用 `TIMESTAMP_UTC` 判斷），其餘標記為重複、下游要濾掉

跟之前的原則一樣，**不刪除原始資料，只新增欄位**：`is_template`、`is_duplicate`、`is_earliest_copy`。下游要用「乾淨」資料時自己篩：

```
keep = (~is_template) & (~is_duplicate | is_earliest_copy)
```

這一版字典直接用 `merged_candidates` 的 `phrase` 欄位，**不使用 `template_type`**（還沒人工複核分類，先當成同一組樣板詞處理）。同一秒內如果有好幾篇文章的 `TIMESTAMP_UTC` 完全相同、分不出先後，就用 pandas `idxmin()` 選到誰算誰，不特別處理 tie-break。

#### 1. 路徑與參數設定

In [5]:
import os

YEARS = [2000, 2006, 2012, 2018, 2024]

dict_source_xlsx = r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\all_years_template_phrase_candidates.xlsx'

output_dir = r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output'
os.makedirs(output_dir, exist_ok=True)

FREQ_THRESHOLD = 5               # 不同文章數達到這個門檻，且沒被字典抓到，才算「高頻轉載」

# 是否真的把每年完整的 snippet 檔案（含新增欄位）寫出來——單一年份可能到 1~2 GB，
# 預設關閉，先看統計數字，確認沒問題後再打開
WRITE_FULL_FLAGGED_CSV = False


def snippet_csv_path(year):
    return (
        r'C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments'
        rf'\output\llm_ready_整年\{year}\llm_ready_data_{year}_context50_snippetlevel.csv'
    )


def flagged_output_path(year):
    return os.path.join(output_dir, f'{year}_snippet_dedup_flagged.csv')


print('dict_source_xlsx:', dict_source_xlsx)
for y in YEARS:
    print(y, '->', snippet_csv_path(y))

dict_source_xlsx: C:\Users\user\Desktop\ravenpack\RP_SCrisk\3_template_Content_process\output\all_years_template_phrase_candidates.xlsx
2000 -> C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2000\llm_ready_data_2000_context50_snippetlevel.csv
2006 -> C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2006\llm_ready_data_2006_context50_snippetlevel.csv
2012 -> C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2012\llm_ready_data_2012_context50_snippetlevel.csv
2018 -> C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2018\llm_ready_data_2018_context50_snippetlevel.csv
2024 -> C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2024\llm_ready_data_2024_context50_snippetlevel.csv

#### 2. 讀入資料驅動字典（`merged_candidates`）

In [6]:
import pandas as pd

phrases_df = pd.read_excel(dict_source_xlsx, sheet_name='merged_candidates')
phrases_df = phrases_df.dropna(subset=['phrase'])
phrases_df = phrases_df[phrases_df['phrase'].str.strip() != '']

phrase_list = list(zip(phrases_df['phrase'], phrases_df['phrase'].str.split().apply(len)))

print(f"讀入 {len(phrase_list):,} 個候選片語")
print(f"片語長度：最短 {min(wc for _, wc in phrase_list)} 字，最長 {max(wc for _, wc in phrase_list)} 字，"
      f"平均 {sum(wc for _, wc in phrase_list) / len(phrase_list):.1f} 字")

讀入 1,842 個候選片語
片語長度：最短 4 字，最長 102 字，平均 11.4 字


#### 3. 比對 + 去重邏輯

模板判斷改成「只要命中字典裡任何一個片語就算模板」，不用比例門檻。`process_year` 對單一年份做完整流程：算 `dup_article_count`、逐一唯一片段跑字典比對、標 `is_template`/`is_duplicate`，再對 `is_duplicate` 的片段各自找最早出現的 `file_name`，標記 `is_earliest_copy`。

實測用完整的 1,842 個候選片語比對，2024 年 939,278 筆唯一片段大約需要 10 分鐘；其他年份唯一片段數較少，時間會等比例縮短。

In [7]:
def process_year(year):
    """對單一年份的 snippet 全量資料做模板剔除 + 轉載去重標記，回傳 (df_flagged, summary_dict)"""
    src = snippet_csv_path(year)
    print(f"\n===== {year} 年 =====")
    print(f"讀取: {src}")
    df = pd.read_csv(src)
    print(f"總列數: {len(df):,}")

    # 只對「唯一文字」算一次，避免對幾百萬列重複計算
    text_stats = df.groupby('llm_input_text').agg(
        dup_article_count=('file_name', 'nunique'),
    ).reset_index()

    # 計算每個片段是否為高頻轉載（不同文章數 >= FREQ_THRESHOLD），並標記
    text_stats['freq_flag'] = text_stats['dup_article_count'] >= FREQ_THRESHOLD
    # 只要命中字典裡任何一個片語就算模板，不用比例門檻（逐列套用，結果存回欄位）
    text_stats['is_template'] = text_stats['llm_input_text'].apply(
        lambda text: any(f" {phrase} " in f" {text} " for phrase, _ in phrase_list)
    )

    text_stats['is_duplicate'] = text_stats['freq_flag'] & (~text_stats['is_template'])

    print(f"唯一片段數: {len(text_stats):,}")
    print(f"  is_template（字典命中，不論次數）: {text_stats['is_template'].sum():,}")
    print(f"  is_duplicate（非模板但高頻，需去重留最早）: {text_stats['is_duplicate'].sum():,}")

    # 對 is_duplicate 的片段，找每個片段最早出現的 file_name（同一秒分不出先後就用 idxmin 選到誰算誰）
    dup_texts = set(text_stats.loc[text_stats['is_duplicate'], 'llm_input_text'])
    if dup_texts:
        df_dup = df.loc[
            df['llm_input_text'].isin(dup_texts),
            ['llm_input_text', 'file_name', 'TIMESTAMP_UTC']].copy()
        df_dup['TIMESTAMP_UTC'] = pd.to_datetime(df_dup['TIMESTAMP_UTC'],
                                                 errors='coerce')
        earliest_idx = df_dup.groupby(
            'llm_input_text')['TIMESTAMP_UTC'].idxmin()
        earliest_file_by_text = df_dup.loc[earliest_idx].set_index(
            'llm_input_text')['file_name']
    else:
        earliest_file_by_text = pd.Series(dtype=object)

    lookup = text_stats[[
        'llm_input_text', 'dup_article_count', 'is_template', 'is_duplicate'
    ]].copy()
    lookup['earliest_file'] = lookup['llm_input_text'].map(
        earliest_file_by_text)

    df = df.merge(lookup, on='llm_input_text', how='left')
    df['is_earliest_copy'] = True
    dup_mask = df['is_duplicate']
    df.loc[dup_mask,
           'is_earliest_copy'] = df.loc[dup_mask,
                                        'file_name'] == df.loc[dup_mask,
                                                               'earliest_file']
    df = df.drop(columns=['earliest_file'])

    keep_mask = (~df['is_template']) & (~df['is_duplicate']
                                        | df['is_earliest_copy'])
    n_keep = keep_mask.sum()
    print(f"最終保留列數（非模板 且（非重複 或 是該片段最早那篇）): {n_keep:,} / {len(df):,}"
          f"（{n_keep / len(df):.1%}）")

    summary = {
        'year': year,
        'total_rows': len(df),
        'unique_texts': len(text_stats),
        'template_texts': int(text_stats['is_template'].sum()),
        'duplicate_texts': int(text_stats['is_duplicate'].sum()),
        'keep_rows': int(n_keep),
        'keep_ratio': round(n_keep / len(df), 4),
    }
    return df, summary

#### 4. 逐年套用

`WRITE_FULL_FLAGGED_CSV = False`（見 cell 2）時只算統計數字、不寫出完整檔案；確認統計數字合理之後，把它改成 `True` 重跑這個 cell，才會把每年完整的 snippet 檔案（含新增的 `is_template`/`is_duplicate`/`is_earliest_copy` 欄位）寫成 `{year}_snippet_dedup_flagged.csv`——單一年份可能到 1~2 GB，記得留意硬碟空間。

In [8]:
import gc

summaries = []
for y in YEARS:
    df_flagged, summary = process_year(y)
    summaries.append(summary)

    if WRITE_FULL_FLAGGED_CSV:
        out_path = flagged_output_path(y)
        df_flagged.to_csv(out_path, index=False, encoding='utf-8-sig')
        print(f"已輸出: {out_path}")

    del df_flagged
    gc.collect()

summary_df = pd.DataFrame(summaries)
summary_df


===== 2000 年 =====
讀取: C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2000\llm_ready_data_2000_context50_snippetlevel.csv
總列數: 248,628
唯一片段數: 164,371
  is_template（字典命中，不論次數）: 95,953
  is_duplicate（非模板但高頻，需去重留最早）: 873
最終保留列數（非模板 且（非重複 或 是該片段最早那篇）): 79,900 / 248,628（32.1%）

===== 2006 年 =====
讀取: C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2006\llm_ready_data_2006_context50_snippetlevel.csv
總列數: 806,098
唯一片段數: 357,040
  is_template（字典命中，不論次數）: 179,284
  is_duplicate（非模板但高頻，需去重留最早）: 4,139
最終保留列數（非模板 且（非重複 或 是該片段最早那篇）): 232,764 / 806,098（28.9%）

===== 2012 年 =====
讀取: C:\Users\user\Desktop\ravenpack\RP_SCrisk\2_calculate_supply_chain_risk_and_extract_segments\output\llm_ready_整年\2012\llm_ready_data_2012_context50_snippetlevel.csv
總列數: 889,880
唯一片段數: 422,934
  is_template（字典命中，不論次數）: 149,192
  is_duplicate（非模板但高頻，需去重留最早）: 6,044
最終保留列數（非模板 且（非重複 或 是該片段最早那篇）

,year,total_rows,unique_texts,template_texts,duplicate_texts,keep_rows,keep_ratio
0,2000,248628,164371,95953,873,79900,0.3214
1,2006,806098,357040,179284,4139,232764,0.2888
2,2012,889880,422934,149192,6044,364650,0.4098
3,2018,1222939,517863,162314,10393,460157,0.3763
4,2024,2207756,939278,242679,27796,913515,0.4138


#### 5. 跨年度比較

In [9]:
print(summary_df.to_string(index=False))
summary_df

 year  total_rows  unique_texts  template_texts  duplicate_texts  keep_rows  keep_ratio
 2000      248628        164371           95953              873      79900      0.3214
 2006      806098        357040          179284             4139     232764      0.2888
 2012      889880        422934          149192             6044     364650      0.4098
 2018     1222939        517863          162314            10393     460157      0.3763
 2024     2207756        939278          242679            27796     913515      0.4138


,year,total_rows,unique_texts,template_texts,duplicate_texts,keep_rows,keep_ratio
0,2000,248628,164371,95953,873,79900,0.3214
1,2006,806098,357040,179284,4139,232764,0.2888
2,2012,889880,422934,149192,6044,364650,0.4098
3,2018,1222939,517863,162314,10393,460157,0.3763
4,2024,2207756,939278,242679,27796,913515,0.4138
